<a href="https://colab.research.google.com/github/cyberirishman/5-day-AI-Cyber/blob/main/Lab3b_Normalization_Homes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Day 2 · Lab 3b — Normalization: Put Every Column on the Same Ruler

**AI for Cybersecurity Professionals · Day 2: AI for Defense**

In **Lab 3a** we *cleaned* the house data. Now the data is spotless — but there's still a hidden
problem that can blind a model, and it has nothing to do with dirt. It's about **scale**.

- `bedrooms` runs 1-6
- `age_years` runs 0-100
- `sqft` runs from ~500 to ~3,800

Many models measure how "similar" two houses are using **distance**. When one column is
measured in *thousands* and another in *single digits*, the big-number column drowns out the
others — even if the small-number columns are what really decide the price.

This lab isolates **one** variable: scaling. We hold cleaning constant (we start from the
already-clean data) so the *only* thing that changes is whether we normalize. That makes it
crystal-clear what normalization does.

### What you'll do in each step

| Step | What happens |
|---|---|
| **1** | Load the libraries, including **KNN** (a distance-based model) and **MinMaxScaler** (the normalizer). |
| **2** | Load the already-clean house data **straight from GitHub** — nothing to upload. |
| **3** | See the scale mismatch: how far each column actually spans. |
| **4** | Hand-work the distance between two houses on **raw** numbers — and watch `sqft` swallow it. |
| **5** | **Normalize** every column to a 0-1 ruler. |
| **6** | Re-do the same distance on **scaled** numbers — now all three features count. |
| **7** | Train a KNN model **without** scaling and measure its error. |
| **8** | Train the **same** model **with** scaling. Only one thing changed. |
| **9** | Chart the two errors side by side. |

> **Nothing to download or upload.** Open this notebook from its **Open in Colab** badge above
> and the data loads itself from GitHub.

> No prior Python needed — every block of code is explained in the comments (the grey text
> after a `#`).

## Step 1 — Set up our tools

In [ ]:
# ---- The usual toolboxes ------------------------------------------------------
#   pandas     : tables (a table = a "DataFrame")
#   numpy      : fast maths on numbers
#   matplotlib : draws charts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- The model we will use: k-Nearest-Neighbors (KNN) -------------------------
# KNN is deliberately simple, and that is why it shows the scaling problem so clearly.
# To predict a house's price it finds the k most SIMILAR houses it already knows
# (the "nearest" ones by distance) and averages their prices. Everything therefore
# depends on how "distance" is measured -- which is exactly what scaling changes.
from sklearn.neighbors import KNeighborsRegressor

from sklearn.model_selection import train_test_split   # splits data into train vs. test
from sklearn.metrics import mean_squared_error         # measures how wrong the model is

# ---- The normalizer -----------------------------------------------------------
# MinMaxScaler rewrites every column so its smallest value becomes 0 and its
# largest becomes 1. That is the "same ruler" idea, in one tool.
from sklearn.preprocessing import MinMaxScaler

pd.set_option("display.max_columns", 20)
print("Libraries loaded. Ready to go.")

## Step 2 — Load the already-clean data

We reuse `homes_clean.csv` from Lab 3a — read **straight from the internet**, so there is
nothing to upload and no file path to get wrong on Mac, Windows or Linux. There is no cleaning
in this lab, and that is the whole point: cleaning is held constant so scaling is the only
thing that changes.

In [ ]:
# ---- Where the data comes from ------------------------------------------------
# Note this is the RAW GitHub address (raw.githubusercontent.com), which returns the
# file itself. A normal github.com link returns a web PAGE and pandas cannot read it.
DATA_URL  = "https://raw.githubusercontent.com/cyberirishman/5-day-AI-Cyber/main/homes_clean.csv"
DATA_FILE = "homes_clean.csv"    # only used by the offline fallbacks below

import os   # lets us ask the computer whether a file exists on disk

# ---- Same loader as Lab 3a: internet first, then local file, then upload box ----
def load_csv(url="", fname=""):
    """Load the CSV from a URL, or a local file, or (on Colab) an upload box."""
    if url:
        try:
            return pd.read_csv(url)
        except Exception as problem:
            print("Could not read from the internet:", problem)
            print("Falling back to a local copy...")
    for path in [fname, os.path.join("data", fname)]:
        if fname and os.path.exists(path):
            return pd.read_csv(path)
    from google.colab import files
    uploaded = files.upload()
    return pd.read_csv(list(uploaded.keys())[0])

# ---- Load it ------------------------------------------------------------------
homes = load_csv(DATA_URL, DATA_FILE)

print("Loaded", len(homes), "clean houses.")
homes.head()    # first 5 rows, so we can see the columns we are about to scale

**What you should see:** `Loaded 1000 clean houses.` and a tidy four-column table with no `$`
signs, no blanks and no silly values — Lab 3a already dealt with all of that.

## Step 3 — See the scale mismatch

Look at the min and max of each column. Notice how much bigger `sqft` is than the others.

In [ ]:
# ---- How far does each column actually span? -----------------------------------
# .agg(["min", "max"]) applies BOTH of those calculations to each column at once
# and returns a small summary table.
ranges = homes[["bedrooms", "sqft", "age_years"]].agg(["min", "max"])
print(ranges)

# int(...) just chops off the decimal point so the sentence reads cleanly.
print("\nsqft spans about", int(homes["sqft"].max() - homes["sqft"].min()),
      "-- while bedrooms spans only", int(homes["bedrooms"].max() - homes["bedrooms"].min()))

## Step 4 — Hand-worked example: distance between two houses (RAW)

Take two houses that differ by **2 bedrooms, 500 sqft, and 10 years**. Distance-based models
square each difference and add them up. Watch what happens.

In [ ]:
# ---- Two made-up houses with easy round differences ------------------------------
# The { } braces make a "dictionary": a set of name -> value pairs.
# We invent these two so the arithmetic is easy to follow by hand.
house_A = {"bedrooms": 3, "sqft": 2000, "age_years": 40}
house_B = {"bedrooms": 5, "sqft": 2500, "age_years": 50}

# ---- How different are they, feature by feature? ---------------------------------
d_bed  = house_B["bedrooms"]  - house_A["bedrooms"]     # = 2
d_sqft = house_B["sqft"]      - house_A["sqft"]         # = 500
d_age  = house_B["age_years"] - house_A["age_years"]    # = 10

print("Raw differences:  bedrooms {}, sqft {}, age {}".format(d_bed, d_sqft, d_age))

# ---- Squaring is what distance maths does ----------------------------------------
# ** means "to the power of", so d_bed**2 is d_bed squared.
# Squaring is why a big raw number does not just win -- it wins overwhelmingly.
print("Squared:          bedrooms {}, sqft {}, age {}".format(d_bed**2, d_sqft**2, d_age**2))

total = d_bed**2 + d_sqft**2 + d_age**2
print("sqft accounts for {:.2%} of the total distance".format(d_sqft**2 / total))

`500² = 250,000` utterly dwarfs `2² = 4` and `10² = 100`. Over **99.9%** of the "distance"
comes from `sqft` alone. The model is effectively **blind to bedrooms and age** — even though,
in our data, bedrooms is the *strongest* driver of price. That's the trap.

## Step 5 — Normalize: rescale every column to 0-1

**Min-max normalization** rewrites each value as `(value − min) / (max − min)`, so the smallest
value in a column becomes 0 and the largest becomes 1. Let's see the same houses after scaling.

In [ ]:
# ---- Teach the scaler what "smallest" and "largest" mean --------------------------
# .fit() looks at the real data and remembers each column's min and max.
# (For this illustration we fit on all the houses. In Step 8, where we actually train
#  a model, we will be stricter about that -- see the golden rule there.)
scaler = MinMaxScaler().fit(homes[["bedrooms", "sqft", "age_years"]])

# ---- Put our two example houses into a small table --------------------------------
rows = pd.DataFrame([house_A, house_B], index=["house_A", "house_B"])

# .transform() applies the (value - min) / (max - min) formula to every value.
# We wrap the result back into a DataFrame so it prints with proper column names.
scaled = pd.DataFrame(scaler.transform(rows), index=rows.index,
                      columns=["bedrooms", "sqft", "age_years"]).round(2)

print("RAW values:\n", rows, "\n")
print("SCALED to 0-1:\n", scaled)

## Step 6 — Re-do the distance (SCALED)

In [ ]:
# ---- The same subtraction, but on the 0-1 versions ---------------------------------
sd = scaled.loc["house_B"] - scaled.loc["house_A"]   # .loc[name] picks a row by its label
print("Scaled differences:  bedrooms {:.2f}, sqft {:.2f}, age {:.2f}".format(
      sd["bedrooms"], sd["sqft"], sd["age_years"]))

# ---- Square them and see who contributes what --------------------------------------
sq = sd**2            # squares every value at once
total_s = sq.sum()    # add them all up

for f in ["bedrooms", "sqft", "age_years"]:
    print("  {:<9} contributes {:.1%} of the scaled distance".format(f, sq[f] / total_s))

Now all three features have a real say. `bedrooms` — the feature that actually decides
the price — finally counts instead of being buried under `sqft`'s big raw numbers.

## Step 7 — Prove it on a model: KNN **without** scaling

Train a k-Nearest-Neighbors model on the **raw** columns and measure its error (RMSE = typical
dollars off; lower is better).

In [ ]:
# ---- Split the data into inputs and answer -----------------------------------------
X = homes[["bedrooms", "sqft", "age_years"]]   # inputs (the "features")
y = homes["price"]                             # the thing we predict (the "label")

# Hold back 20% of the houses so we grade the model on examples it never trained on.
# random_state=1 fixes which houses are held back, so everyone gets the same number.
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=1)

# ---- Train KNN on the RAW numbers ----------------------------------------------------
# n_neighbors=5 means: to price a house, find the 5 most similar houses and average them.
knn_raw = KNeighborsRegressor(n_neighbors=5).fit(X_tr, y_tr)

# ** 0.5 is a square root: it turns MSE into RMSE, which is back in dollars.
rmse_raw = mean_squared_error(y_te, knn_raw.predict(X_te)) ** 0.5
print("UNSCALED KNN  RMSE: ${:,.0f}".format(rmse_raw))

## Step 8 — The SAME model, **with** scaling

Only one thing changes: we scale the features to 0-1 first.

> **Golden rule:** fit the scaler on the **training** data only, then apply it to the test data.
> Otherwise the test set "leaks" into training and your score is a lie.

In [ ]:
# ---- Learn the min/max from the TRAINING data only ------------------------------------
# This is the golden rule in code. If we fitted on all the data, the scaler would have
# peeked at the test houses, and our score would flatter the model.
scaler = MinMaxScaler().fit(X_tr)

X_tr_s = scaler.transform(X_tr)   # apply the scaling to the training houses
X_te_s = scaler.transform(X_te)   # apply the SAME scaling to the test houses

# ---- Exactly the same model, exactly the same houses, only the ruler changed ------------
knn_scaled = KNeighborsRegressor(n_neighbors=5).fit(X_tr_s, y_tr)
rmse_scaled = mean_squared_error(y_te, knn_scaled.predict(X_te_s)) ** 0.5

print("UNSCALED KNN  RMSE: ${:,.0f}".format(rmse_raw))
print("SCALED   KNN  RMSE: ${:,.0f}".format(rmse_scaled))
print("\nScaling cut the typical error by {:.0%} -- and NOTHING else changed.".format(
      1 - rmse_scaled / rmse_raw))

**What you should see:** about **$27,600** unscaled against about **$16,300** scaled — roughly a
**41% cut in typical error**. Same model, same houses, same split. The *only* difference is
which ruler the columns were measured on.

## Step 9 — See it: unscaled vs. scaled error

In [ ]:
# ---- A two-bar chart: the whole lab in one picture ---------------------------------
fig, ax = plt.subplots(figsize=(5, 4))

# .bar() takes the labels for the bars and their heights.
bars = ax.bar(["Unscaled", "Scaled"], [rmse_raw, rmse_scaled],
              color=["#C0392B", "#1E5199"])

ax.set_ylabel("Test RMSE  (typical $ error)")
ax.set_title("Same KNN model - scaling is the only change")

# ---- Print the actual dollar figure on top of each bar --------------------------------
# zip() walks through both lists together, one bar and one value at a time.
for b, v in zip(bars, [rmse_raw, rmse_scaled]):
    ax.text(b.get_x() + b.get_width()/2, v, "${:,.0f}".format(v),
            ha="center", va="bottom")

plt.tight_layout()
plt.show()

## Wrap-up

- The house data was already clean, yet an **unscaled** distance model still did badly —
  because `sqft`'s big raw numbers drowned out the features that actually set the price.
- **Normalization** (min-max to 0-1) put every column on the same ruler, and the *same* model's
  error dropped sharply.
- **This is why neural networks need scaled inputs too** — the same "big-number features
  dominate" effect shows up there. We'll see a neural net next (§2.8.4).
- **Rule of thumb:** always fit your scaler on training data only, then reuse it on test/live
  data.